[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/05_attention.ipynb)

# 🔴 Hard: Softmax Attention

*Attention & Transformers*
Implement **scaled dot-product attention** as a plain function.

$$\text{Attention}(Q, K, V) = \operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

### Signature
`scaled_dot_product_attention(q, k, v, mask=None)`

| tensor | shape | notes |
|---|---|---|
| `q` | `(B, T_q, d_k)` | queries |
| `k` | `(B, T_k, d_k)` | keys — same feature dim as `q` |
| `v` | `(B, T_k, d_v)` | values — `d_v` may differ from `d_k` |
| `mask` | broadcastable to `(..., T_q, T_k)` | boolean, `True` = **attend**, `False` = **block** |
| returns | `(B, T_q, d_v)` | |

### Rules
- No `jax.nn.dot_product_attention`, no `nnx.MultiHeadAttention`
- `jax.nn.softmax` is allowed and encouraged (it subtracts the row max for you)
- `T_q` and `T_k` are independent — do not assume a square score matrix
- Only the **last two** axes are contracted, so the identical code must also
  accept `(B, H, T, D_h)` inputs. Use `jnp.swapaxes(k, -1, -2)`, never a
  hard-coded `transpose(0, 2, 1)`
- Masked positions are removed **before** the softmax, not zeroed after it

### Why the $1/\sqrt{d_k}$ is not cosmetic
Take $q, k$ with i.i.d. zero-mean unit-variance entries. Their dot product is a
sum of $d_k$ independent unit-variance terms, so

$$\operatorname{Var}(q \cdot k) = d_k, \qquad \operatorname{std}(q \cdot k) = \sqrt{d_k}$$

At $d_k = 64$ the raw logits have standard deviation $8$, so across a few hundred
keys the gap between the largest logit and the mean runs to about $23$ (measured
over 300 trials: 10th–90th percentile $19$–$28$).

Concretely, at that scale one key takes roughly **59%** of the attention mass;
after dividing by $\sqrt{64}$ it takes **3%**. So the unscaled version is not
literally one-hot, but it is sharply peaked — and the Jacobian of softmax,
$\operatorname{diag}(p) - pp^\top$, shrinks toward zero as $p$ concentrates, so
the layer passes progressively less gradient the sharper it gets. Dividing by
$\sqrt{d_k}$ pulls the logit variance back to $1$ regardless of head width,
which is exactly why you can widen heads without retuning the initialisation.

Interviewers probe two things here. First that the scale is present at all.
Second — the near-miss that actually separates candidates — that it is
$\sqrt{d_k}$, the **per-head key** dim, and not $\sqrt{d_{model}}$. With $H$
heads those dims differ by a factor of $H$, so the two scale factors differ by
$\sqrt{H}$: at $H = 16$ you would be shrinking every logit by an extra $4\times$
and the attention would come out close to uniform.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def scaled_dot_product_attention(q, k, v, mask=None):
    """softmax(q k^T / sqrt(d_k)) v

    Args:
        q:    (..., T_q, d_k)
        k:    (..., T_k, d_k)
        v:    (..., T_k, d_v)
        mask: optional boolean array broadcastable to (..., T_q, T_k);
              True means "attend here", False means "block".

    Returns:
        (..., T_q, d_v)
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

# What the scaling actually buys you: logit spread vs head width.
# 64 queries over 512 keys, so the sample statistics settle near their limits.
for d_k in (8, 64, 512):
    kq, kk = jax.random.split(jax.random.key(d_k))
    q = jax.random.normal(kq, (1, 64, d_k))
    k = jax.random.normal(kk, (1, 512, d_k))
    raw = (q @ jnp.swapaxes(k, -1, -2))[0]              # (64, 512)
    scaled = raw / jnp.sqrt(float(d_k))
    p_raw = jax.nn.softmax(raw, axis=-1)
    p_scaled = jax.nn.softmax(scaled, axis=-1)
    print(f"d_k={d_k:4d}  sqrt(d_k)={float(d_k) ** 0.5:6.2f}"
          f"  logit std raw={raw.std():6.2f} scaled={scaled.std():4.2f}"
          f"  mean max prob raw={p_raw.max(-1).mean():.3f}"
          f" scaled={p_scaled.max(-1).mean():.3f}")

# Cross shapes: 3 queries attending over 5 keys, values 8-dim.
q = jax.random.normal(jax.random.key(1), (2, 3, 16))
k = jax.random.normal(jax.random.key(2), (2, 5, 16))
v = jax.random.normal(jax.random.key(3), (2, 5, 8))
print("out:", scaled_dot_product_attention(q, k, v).shape)   # (2, 3, 8)

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("attention")

# hint("attention")      # stuck? nudge without the answer
# solution("attention")  # spoiler: the reference implementation